# Model 4 - Gingival Inflammation - Training

Detect inflamed-gum regions (gingivitis) on intraoral / mobile photos. Single class.

**Architecture:** YOLO11s detect | **imgsz:** 640 | **Data:** sampling/gingivitis + Pranta/gum-disease (CC BY 4.0)

Run on Kaggle (GPU T4, Internet On) or locally with a CUDA GPU. For the full Kaggle pipeline (dataset download + merge), use `notebooks/train_model4_kaggle.ipynb`.

## 1. Environment

In [ ]:
!pip -q install -U ultralytics
import torch, ultralytics
print("torch", torch.__version__, "| CUDA", torch.cuda.is_available(), "| GPUs", torch.cuda.device_count())
ultralytics.checks()

## 2. Dataset
Point `DATA` at the merged `data.yaml` (single `gingivitis` class, train/valid/test).

In [ ]:
DATA = "merged/data.yaml"
import yaml
d = yaml.safe_load(open(DATA))
print("classes:", d.get("names"))

## 3. Train
Single GPU avoids the Kaggle 2-GPU DDP crash. Set CUDA_VISIBLE_DEVICES before importing torch there.

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"]="0"
from ultralytics import YOLO
model = YOLO("yolo11s.pt")
results = model.train(
    data=DATA, epochs=120, imgsz=640, batch=16, device=0,
    patience=30, seed=42, project="runs", name="train", exist_ok=True,
    close_mosaic=10, save_period=20, verbose=True)
print("TRAIN_DONE")

## 4. Validate on the test split

In [ ]:
best = "runs/train/weights/best.pt"
m = YOLO(best)
mt = m.val(data=DATA, split="test", imgsz=640, device=0)
print("test mAP50:", round(float(mt.box.map50),4), "| mAP50-95:", round(float(mt.box.map),4),
      "| P:", round(float(mt.box.mp),3), "| R:", round(float(mt.box.mr),3))

## 5. Export + save weights
Copy `best.pt` into this model's `models/` folder.

In [ ]:
import shutil, os
os.makedirs("../models", exist_ok=True)
shutil.copy(best, "../models/best.pt")
try:
    m.export(format='onnx', imgsz=640, opset=12)
    shutil.copy('runs/train/weights/best.onnx', '../models/best.onnx')
except Exception as e:
    print('onnx export skipped:', e)
print("saved -> models/")